In [ ]:
### Imports

import pathlib

import h5py
import matplotlib.pyplot as plt
import numpy as np
import polars as pls
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

import usv_playpen.analyses.neuronal_coactivity_engine as engine

In [ ]:
### Load the data

data_directory = pathlib.Path('/mnt/falkner/Bartul/Data/20250919_152924')

# Load tracking data
tracking_file = next(item for item in data_directory.glob('**/*_translated_rotated_metric.h5'))

with h5py.File(name=tracking_file, mode='r') as track_file:
    mouse_tracks = np.array(track_file['tracks'])
    mouse_track_names = [item.decode('utf-8') for item in list(track_file['track_names'])]
    recording_frame_rate = float(track_file['recording_frame_rate'][()])

# Load vocalization data and filter for male USVs
usv_summary_file = next(item for item in data_directory.glob('**/*_usv_summary.csv'))
usv_summary_data = pls.read_csv(usv_summary_file)
usv_summary_data = usv_summary_data.filter(pls.col('emitter') == mouse_track_names[0])

# Load spike data (in seconds, not tracking frames)
neural_data_dict = {}
for spike_file in data_directory.glob('**/*_good.npy'):
    neural_data_dict[spike_file.stem] = np.load(spike_file)[0, :]

In [ ]:
### Compute Coactivity Metrics for Complex vs. Simple Vocalizations

total_duration = mouse_tracks.shape[0] / recording_frame_rate

# Filter groups
complex_df = usv_summary_data.filter(pls.col('usv_supercategory').is_in([1, 2]))
simple_df = usv_summary_data.filter(pls.col('usv_supercategory') == 5)

print(f"Total Vocalizations: {len(usv_summary_data)} | Complex: {len(complex_df)} | Simple: {len(simple_df)}")

# Set a time window comparable for simple and complex calls (e.g., 30 ms)
WINDOW_S = 0.030 

# Extract count matrices (neurons x trials)
complex_counts = engine.extract_snippet_matrix(
    onsets=complex_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

simple_counts = engine.extract_snippet_matrix(
    onsets=simple_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

# Compute observed metrics
complex_obs = engine.compute_coactivity_metrics(complex_counts)
simple_obs = engine.compute_coactivity_metrics(simple_counts)

# Statistical validation
# A. Bootstrap simple calls to match the sample size of complex calls (n=71)
bootstrap_results = engine.get_bootstrapped_simple_calls(
    simple_counts=simple_counts, 
    n_target=complex_counts.shape[1], 
    n_iterations=1000
)

# B. Circular temporal shuffle for COMPLEX
complex_shuffle = engine.perform_circular_shuffle(
    onsets=complex_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    total_duration=total_duration, 
    window_s=WINDOW_S, 
    min_shift_s=20.0,
    max_shift_s=60.0,
    n_shuffles=1000
)

# C. Circular temporal shuffle for SIMPLE
simple_shuffle = engine.perform_circular_shuffle(
    onsets=simple_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    total_duration=total_duration, 
    window_s=WINDOW_S, 
    min_shift_s=20.0,
    max_shift_s=60.0,
    n_shuffles=1000
)

# Display Summary Results with Empirical P-values
print(f"Analysis Window: {WINDOW_S*1000:.0f}ms | Session Duration: {total_duration:.2f}s")
print("-" * 75)

# Calculate stats for COMPLEX
shuff_rsc_c = complex_shuffle['r_sc']
shuff_sim_c = complex_shuffle['similarity']

complex_p_rsc = np.mean(shuff_rsc_c >= complex_obs['r_sc'])
complex_p_sim = np.mean(shuff_sim_c >= complex_obs['similarity'])
complex_z_rsc = (complex_obs['r_sc'] - np.mean(shuff_rsc_c)) / np.std(shuff_rsc_c)
complex_z_sim = (complex_obs['similarity'] - np.mean(shuff_sim_c)) / np.std(shuff_sim_c)

print(f"COMPLEX (n={complex_counts.shape[1]}):")
print(f"  r_sc:       Obs={complex_obs['r_sc']:.4f} | Shuff={np.mean(shuff_rsc_c):.4f} | p={complex_p_rsc:.4f} (Z={complex_z_rsc:.2f})")
print(f"  Similarity: Obs={complex_obs['similarity']:.4f} | Shuff={np.mean(shuff_sim_c):.4f} | p={complex_p_sim:.4f} (Z={complex_z_sim:.2f})")

print("-" * 75)

# Calculate stats for SIMPLE
boot_rsc = bootstrap_results['r_sc']
boot_sim = bootstrap_results['similarity']
shuff_rsc_s = simple_shuffle['r_sc']
shuff_sim_s = simple_shuffle['similarity']

simple_p_rsc = np.mean(shuff_rsc_s >= simple_obs['r_sc'])
simple_p_sim = np.mean(shuff_sim_s >= simple_obs['similarity'])
simple_z_rsc = (simple_obs['r_sc'] - np.mean(shuff_rsc_s)) / np.std(shuff_rsc_s)
simple_z_sim = (simple_obs['similarity'] - np.mean(shuff_sim_s)) / np.std(shuff_sim_s)

print(f"SIMPLE (n={simple_counts.shape[1]}):")
print(f"  r_sc:       Boot={np.mean(boot_rsc):.4f} | Shuff={np.mean(shuff_rsc_s):.4f} | p={simple_p_rsc:.4f} (Z={simple_z_rsc:.2f})")
print(f"  Similarity: Boot={np.mean(boot_sim):.4f} | Shuff={np.mean(shuff_sim_s):.4f} | p={simple_p_sim:.4f} (Z={simple_z_sim:.2f})")

In [ ]:
# 1. Prepare Unified Null Distributions
# Combining shuffles from both groups to create a "Global Null"
unified_shuff_rsc = np.concatenate([complex_shuffle['r_sc'], simple_shuffle['r_sc']])
unified_shuff_sim = np.concatenate([complex_shuffle['similarity'], simple_shuffle['similarity']])

# 2. Calculate thresholds (99th Percentile)
rsc_99 = np.percentile(unified_shuff_rsc, 99)
sim_99 = np.percentile(unified_shuff_sim, 99)

# 3. Define values for plotting lines
# Using observed complex and bootstrapped mean for simple (matched sample size)
complex_val_rsc = complex_obs['r_sc']
simple_val_rsc = np.mean(bootstrap_results['r_sc'])

complex_val_sim = complex_obs['similarity']
simple_val_sim = np.mean(bootstrap_results['similarity'])

# 4. Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Pairwise Correlation (r_sc) ---
ax = axes[0]
ax.hist(unified_shuff_rsc, bins=30, color='grey', alpha=0.5, label='Shuffled Null')
ax.axvline(rsc_99, color='black', linestyle='--', linewidth=2, label='99th Percentile')
ax.axvline(complex_val_rsc, color='crimson', linewidth=3, label=f'Complex (Obs={complex_val_rsc:.4f})')
ax.axvline(simple_val_rsc, color='dodgerblue', linewidth=3, label=f'Simple (Boot={simple_val_rsc:.4f})')

ax.set_title(f'Pairwise Correlation ($r_{{sc}}$) - {WINDOW_S*1000:.0f}ms window', fontsize=14)
ax.set_xlabel('Mean $r_{sc}$', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Plot 2: Cosine Similarity ---
ax = axes[1]
ax.hist(unified_shuff_sim, bins=30, color='grey', alpha=0.5, label='Shuffled Null')
ax.axvline(sim_99, color='black', linestyle='--', linewidth=2, label='99th Percentile')
ax.axvline(complex_val_sim, color='crimson', linewidth=3, label=f'Complex (Obs={complex_val_sim:.4f})')
ax.axvline(simple_val_sim, color='dodgerblue', linewidth=3, label=f'Simple (Boot={simple_val_sim:.4f})')

ax.set_title(f'Population Similarity (Cosine) - {WINDOW_S*1000:.0f}ms window', fontsize=14)
ax.set_xlabel('Mean Cosine Similarity', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()